# Scrollie — MuscleMap WB: Water vs Fat Fraction

Three panels side by side for each slice:
- **Left**: original water image (no overlay)
- **Centre**: water image + MuscleMap WB segmentation overlay
- **Right**: fat-fraction image + MuscleMap WB segmentation overlay

In [ ]:
import glob
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import SimpleITK as sitk
from ipywidgets import IntSlider, Dropdown, VBox
import ipywidgets as widgets
from IPython.display import display

In [ ]:
WATER_SEG_DIR   = os.path.join('..', 'musclemap_wb_water_segs')
FATFRAC_SEG_DIR = os.path.join('..', 'musclemap_wb_fat_fraction_segs')
DATA_ROOT       = os.path.join('..', 'myosegmenTUM')

LABEL_MAP = {
    7101: 'Vastus_Lateralis_L',
    7102: 'Vastus_Lateralis_R',
    7111: 'Vastus_Intermedius_L',
    7112: 'Vastus_Intermedius_R',
    7121: 'Vastus_Medialis_L',
    7122: 'Vastus_Medialis_R',
    7131: 'Rectus_Femoris_L',
    7132: 'Rectus_Femoris_R',
    7141: 'Sartorius_L',
    7142: 'Sartorius_R',
    7151: 'Gracilis_L',
    7152: 'Gracilis_R',
    7161: 'Semimembranosus_L',
    7162: 'Semimembranosus_R',
    7171: 'Semitendinosus_L',
    7172: 'Semitendinosus_R',
    7181: 'Biceps_Femoris_L',
    7182: 'Biceps_Femoris_R',
    7201: 'Adductor_Magnus_L',
    7202: 'Adductor_Magnus_R',
}

water_segs = sorted(glob.glob(os.path.join(WATER_SEG_DIR, '*_dseg.nii.gz')))

entries = {}
for ws in water_segs:
    fname = os.path.basename(ws)
    m = re.match(r'(.+)_WATER_(stack\d+)_dseg\.nii\.gz', fname)
    if not m:
        continue
    subject = m.group(1)
    stack   = m.group(2)
    label   = f'{subject}_WATER_{stack}'
    entries[label] = {
        'subject':   subject,
        'stack':     stack,
        'water_seg': ws,
        'ff_seg':    os.path.join(FATFRAC_SEG_DIR, f'{subject}_FATFRACTION_{stack}_dseg.nii.gz'),
        'water_img': os.path.join(DATA_ROOT, subject, 'ImageData',
                                  f'{subject}_WATER', f'{subject}_WATER_{stack}.nii'),
        'ff_img':    os.path.join(DATA_ROOT, subject, 'ImageData',
                                  f'{subject}_FATFRACTION', f'{subject}_FATFRACTION_{stack}.nii'),
    }

print(f'Found {len(entries)} water stacks')
for lbl, e in entries.items():
    flags = [k for k, p in [('no ff seg', e['ff_seg']), ('no ff img', e['ff_img'])]
             if not os.path.exists(p)]
    if flags:
        print(f'  {lbl}: {", ".join(flags)}')

In [ ]:
_cmap = plt.colormaps['tab20']

def _label_color(label_id):
    """Deterministic color per label ID so colours match across panels."""
    return _cmap(label_id % 20 / 20)

def build_overlay(seg_arr):
    present = sorted(k for k in LABEL_MAP if np.any(seg_arr == k))
    overlay = np.zeros((*seg_arr.shape, 4), dtype=float)
    for k in present:
        c = _label_color(k)
        overlay[seg_arr == k] = [c[0], c[1], c[2], 0.5]
    patches = [
        mpatches.Patch(color=_label_color(k), alpha=0.6, label=LABEL_MAP[k])
        for k in present
    ]
    return overlay, patches

def norm(arr):
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo + 1e-8)

def load_entry(e):
    water_arr = norm(sitk.GetArrayFromImage(sitk.ReadImage(e['water_img'])).astype(float))
    water_seg = sitk.GetArrayFromImage(sitk.ReadImage(e['water_seg']))
    water_ov, water_patches = build_overlay(water_seg)

    ff_arr, ff_ov, ff_patches = None, None, []
    if os.path.exists(e['ff_img']) and os.path.exists(e['ff_seg']):
        ff_arr = norm(sitk.GetArrayFromImage(sitk.ReadImage(e['ff_img'])).astype(float))
        ff_seg = sitk.GetArrayFromImage(sitk.ReadImage(e['ff_seg']))
        ff_ov, ff_patches = build_overlay(ff_seg)

    return water_arr, water_ov, water_patches, ff_arr, ff_ov, ff_patches

In [ ]:
file_dropdown = Dropdown(options=list(entries.keys()), description='Stack:')
slice_slider  = IntSlider(min=0, max=1, step=1, value=0, description='Slice:',
                          layout=widgets.Layout(width='600px'))
out = widgets.Output()

_cache = {}

def get_data(label):
    if label not in _cache:
        data = load_entry(entries[label])
        n_slices = data[0].shape[0]
        slice_slider.max   = n_slices - 1
        slice_slider.value = n_slices // 2
        _cache[label] = data
    return _cache[label]

def render(label, slice_idx):
    water_arr, water_ov, water_patches, ff_arr, ff_ov, ff_patches = get_data(label)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(water_arr[slice_idx], cmap='gray', origin='lower')
    axes[0].set_title(f'Water — slice {slice_idx}')
    axes[0].axis('off')

    axes[1].imshow(water_arr[slice_idx], cmap='gray', origin='lower')
    axes[1].imshow(water_ov[slice_idx], origin='lower')
    axes[1].set_title('Water + WB seg')
    axes[1].legend(handles=water_patches, loc='lower right', fontsize=6, framealpha=0.7)
    axes[1].axis('off')

    if ff_arr is not None:
        axes[2].imshow(ff_arr[slice_idx], cmap='gray', origin='lower')
        axes[2].imshow(ff_ov[slice_idx], origin='lower')
        axes[2].set_title('Fat Fraction + WB seg')
        axes[2].legend(handles=ff_patches, loc='lower right', fontsize=6, framealpha=0.7)
    else:
        axes[2].text(0.5, 0.5, 'No fat fraction data', ha='center', va='center',
                     transform=axes[2].transAxes, fontsize=12)
        axes[2].set_title('Fat Fraction + WB seg')
    axes[2].axis('off')

    fig.suptitle(label, fontsize=10)
    plt.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()

def on_file_change(change):
    _cache.clear()
    get_data(change['new'])
    render(file_dropdown.value, slice_slider.value)

def on_slice_change(change):
    render(file_dropdown.value, change['new'])

file_dropdown.observe(on_file_change, names='value')
slice_slider.observe(on_slice_change, names='value')

if entries:
    get_data(file_dropdown.value)
    render(file_dropdown.value, slice_slider.value)

display(VBox([file_dropdown, slice_slider, out]))